# T7 It2: a candidate-level judge for answer-or-skip (no agreement)

**Loss targeted:** the decision after fine matching. Today's agreement-free rule (bar 8 on the 6 standard queries, else bar 11 on all 27) skips
right places with 4-10 inliers on official and still answers look-alike places with 8-11 inliers on fresh NUDT city photos (404, 505).

**Idea:** judge every candidate, not only the chosen one. Each photo has dozens of fine-matched wrong candidates (look-alike buildings), so the
five seen sets give ~9,600 labelled wrong and ~23,500 right candidates with >= 3 inliers, enough to learn what a chance match looks like from all the
signals available after fine matching (block `train/checks/decision_signals.py`, table from `It1_Signal_Table`). Model: monotone gradient
boosting (P(right) may only rise with inliers, DINO z, hull, coverage), block `train/checks/candidate_judge.py`.

**Decision:** pipeline's choice (most inliers + z, among candidates with >= 3 inliers); answer when P(right) >= tau or inliers >= 12; first on the
6 standard queries, then on all 27 (PC) or one extra query at a time up to 10 (Jetson, M = 5).

**Protocol:** leave one set out: the judge and its cut are fitted on the other four sets only (cut = smallest grid value with 0 wrong at it and at
every larger cut, then one grid step more), and the held-out set is replayed once. Baselines: the same replay with the inlier bars (no
agreement); Jetson: the submitted code's replay (bar 8, second pass, cap 10, M = 5).

In [ ]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))   # one OpenMP thread per allowed core: more threads than cores slowed the judge ~100x
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.candidate_judge import (held_out_probabilities, FEATURES, FEATURES_CTX, add_context, add_features, baseline, decide, judge_probability,
                                          leave_one_set_out, paired_gain, summary)
RES = ROOT / "tests/Test_Iteration_7/results"
SETS = ["official", "nudt101", "nudt202", "nudt404", "nudt505"]
table = pd.concat([pd.read_csv(RES / f"signals/{s}.csv.gz") for s in SETS], ignore_index=True)
table = add_context(add_features(table)).reset_index(drop=True)
lab = table[(table.inliers >= 3)]
print(lab.assign(right=lab.err_sigma <= 4, wrong=lab.err_sigma > 8).groupby("data")[["right", "wrong"]].sum())
PROBS = held_out_probabilities(table, SETS, FEATURES_CTX)        # the five leave-one-set-out judges, trained once and reused below

## Baselines (no agreement)

In [ ]:
base_pc = baseline(table, 8, 11, min_choice=3)
base_jet = baseline(table, 8, 8, jetson=True, m=5)
display(pd.concat({"PC: bar 8, then 27 queries @ 11": summary(base_pc), "Jetson today (M 5, 2nd pass @ 8, cap 10)": summary(base_jet)}))

## Feature groups, leave one set out (PC)

In [ ]:
GROUPS = {
    "inliers + z": ["li", "z"],
    "+ geometry (hull, cover9, spread)": ["li", "z", "hull", "cover9", "spread"],
    "+ fit (scale, angle, shift, resid)": ["li", "z", "hull", "cover9", "spread", "scale_dev", "angle", "shift_px", "resid"],
    "+ counts and confidence": ["li", "z", "hull", "cover9", "spread", "scale_dev", "angle", "shift_px", "resid", "ln22", "ratio", "tfrac", "conf_inl", "hc_ratio"],
    "all (+ rank, coarse, sharpness)": FEATURES,
    "all + query-local uniqueness": FEATURES_CTX,
}
abl = []
for name, feats in GROUPS.items():
    s, _ = leave_one_set_out(table, SETS, feats, override=12, min_choice=3)
    abl.append(s.assign(features=name))
abl = pd.concat(abl)
abl["cell"] = abl.score.map("{:.2f}".format) + " / " + abl.wrong.astype(str) + "w / t" + abl.tau.astype(str)
display(abl.pivot(index="features", columns="held", values="cell").loc[list(GROUPS)][SETS])
lg, _ = leave_one_set_out(table, SETS, FEATURES_CTX, kind="logit", override=12, min_choice=3)
print("logistic regression, all + uniqueness:"); display(lg)
abl.to_csv(RES / "it2_feature_groups_loso.csv", index=False)

## The judge on the PC (all signals + uniqueness): held-out scores and paired gains

In [ ]:
pc_sum, pc_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3)
pc_sum["vs PC baseline"] = pc_sum.held.map(paired_gain(base_pc, pc_rows, SETS))
pc_sum["baseline"] = pc_sum.held.map(lambda s: "%.2f / %dw" % (6 * base_pc[base_pc.data == s].pts.mean(), (base_pc[base_pc.data == s].pts < 0).sum()))
display(pc_sum)
nc_sum, nc_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=None, min_choice=3)
print("without the 12-inlier override:"); display(nc_sum)
cj_sum, cj_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3, choose="judge")
print("choosing the candidate by the judge (not by inliers + z):"); display(cj_sum)
pc_sum.to_csv(RES / "it2_judge_pc_loso.csv", index=False); pc_rows.to_csv(RES / "it2_judge_pc_loso_photos.csv", index=False)

### Which photos change (PC, held-out decisions vs the baseline)

In [ ]:
j = base_pc.set_index(["data", "photo"]).join(pc_rows.set_index(["data", "photo"]), rsuffix="_judge")
changed = j[j.pts != j.pts_judge][["stage", "pts", "inliers", "stage_judge", "pts_judge", "inliers_judge", "p_judge", "err_sigma_judge"]]
display(changed.round(2))
print(changed.groupby(level=0).apply(lambda g: pd.Series(dict(gained=(g.pts_judge > g.pts).sum(), lost=(g.pts_judge < g.pts).sum(),
                                                               wrong_removed=((g.pts < 0) & (g.pts_judge >= 0)).sum()))))

## The judge on the Jetson (M = 5, extra queries one at a time, cap 10)

In [ ]:
jet_sum, jet_rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, override=12, min_choice=3, jetson=True, m=5)
jet_sum["vs Jetson today"] = jet_sum.held.map(paired_gain(base_jet, jet_rows, SETS))
jet_sum["Jetson today"] = jet_sum.held.map(lambda s: "%.2f / %dw" % (6 * base_jet[base_jet.data == s].pts.mean(), (base_jet[base_jet.data == s].pts < 0).sum()))
jet_sum["modelled s / photo"] = (jet_sum.queries * 2.2).round(1)
display(jet_sum)
jet_sum.to_csv(RES / "it2_judge_jetson_loso.csv", index=False); jet_rows.to_csv(RES / "it2_judge_jetson_loso_photos.csv", index=False)

## What the judge looks at (permutation importance on the held-out official set, judge trained on the four NUDT sets)

In [ ]:
from sklearn.inspection import permutation_importance
from train.checks.candidate_judge import labelled
nudt = [s for s in SETS if s != "official"]
_, model = judge_probability(table, nudt, FEATURES_CTX)
off = table[(table.data == "official") & labelled(table)]
imp = permutation_importance(model, off[FEATURES_CTX].to_numpy(float), (off.err_sigma <= 4).to_numpy(), scoring="roc_auc", n_repeats=5, random_state=0)
display(pd.Series(imp.importances_mean, FEATURES_CTX).sort_values(ascending=False).round(4).to_frame("AUC drop"))

## Reading

- Inliers + z alone (what today's bars use) gains nothing on official under this protocol; the gain comes from the inlier geometry (hull,
  coverage), the counts (pairs, inlier ratio) and query-local uniqueness.
- 0 wrong answers on every held-out set, and the judge removes the wrong answers the plain bars make on fresh NUDT 404 (2) and 505 (1, photo 889,
  which every Test_Iteration_6 rule got wrong).
- NUDT 202 loses about 0.6: two right answers at 8 and 11 inliers get P 0.94, just under the cut.
- The cut picked on the training sets is 0.925-0.97, well above the break-even of a -20 / +10 game (~0.69): the margin is for safety, not tuned
  for score.